# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam271/flyrank-ml-internship/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [7]:
%pip -q install duckdb huggingface_hub


In [8]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [9]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [10]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [11]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100


In [15]:
# Experiment 1: 90-day feature window

features_90d = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            -- Most recent 30 days
            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_last30,

            -- Previous 30 days
            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 60 DAY
                     AND f.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev30,

            -- Older 30 days
            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 90 DAY
                     AND f.report_date <= b.end_d - INTERVAL 60 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_old30,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_clicks
                    ELSE 0
                END
            ) AS clk_last30,

            AVG(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_avg_position
                END
            ) AS pos_last30

        FROM {TABLES['fact_daily']} f, bounds b

        WHERE f.report_date > b.end_d - INTERVAL 90 DAY

        GROUP BY 1, 2

        HAVING imp_prev30 >= 100
    )

    SELECT *
    FROM windowed
""").df()

print(f"90-day feature table: {len(features_90d):,} content items")
features_90d.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

90-day feature table: 111,247 content items


,client_hash_id,content_hash_id,imp_last30,imp_prev30,imp_old30,clk_last30,pos_last30
0,client_e547b89c05043229,content_d0fa1bbfbc10caf8,1628.0,2298.0,2070.0,0.0,11.779379
1,client_e547b89c05043229,content_4c1e972bec56132e,7098.0,13712.0,5872.0,44.0,6.959208
2,client_e547b89c05043229,content_64cad58fc02e7605,290.0,251.0,598.0,0.0,48.972195
3,client_e547b89c05043229,content_4e48bd81bb37eb4f,23065.0,21066.0,21394.0,12.0,28.140689
4,client_e547b89c05043229,content_f338440914b1ab00,1460.0,1919.0,2602.0,5.0,20.731918


In [16]:
# Add the new 90-day historical feature

data_90d = features_90d.merge(
    qsignals,
    on="content_hash_id",
    how="left"
)

# Create the same declining label
data_90d["is_declining"] = (
    data_90d["imp_last30"] < 0.8 * data_90d["imp_prev30"]
).astype(int)

# Features including the new historical feature
feature_cols_90d = [
    "imp_prev30",
    "imp_old30",          # NEW FEATURE
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
]

model_data_90d = data_90d.dropna(subset=feature_cols_90d)

print(f"Rows available for modeling: {len(model_data_90d):,}")
print("\nFeatures used:")
print(feature_cols_90d)

Rows available for modeling: 102,203

Features used:
['imp_prev30', 'imp_old30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X = model_data_90d[feature_cols_90d]
y = model_data_90d["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

random_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

random_model.fit(X_train, y_train)

random_pred = random_model.predict(X_test)

print("RANDOM SPLIT")
print(classification_report(y_test, random_pred))

RANDOM SPLIT
              precision    recall  f1-score   support

           0       0.66      0.41      0.51      9389
           1       0.72      0.88      0.79     16162

    accuracy                           0.71     25551
   macro avg       0.69      0.64      0.65     25551
weighted avg       0.70      0.71      0.69     25551



In [ ]:
from sklearn.model_selection import GroupShuffleSplit

X = model_data_90d[feature_cols_90d]
y = model_data_90d["is_declining"]
groups = model_data_90d["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

group_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

group_model.fit(X_train_group, y_train_group)

group_pred = group_model.predict(X_test_group)

print("GROUP SHUFFLE SPLIT — CLIENT LEVEL")
print(classification_report(y_test_group, group_pred))

In [ ]:
# Feature importance from the client-level model

importance = pd.DataFrame({
    "feature": feature_cols_90d,
    "importance": group_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("FEATURE IMPORTANCE")
print(importance.to_string(index=False))

## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [12]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789,1.0,0.022750,0.957216,59.0,59.0,1.000000
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636,14.0,0.017946,0.932994,84.0,462.0,0.181818
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167,3.0,0.162037,0.552469,153.0,185.0,0.827027
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367,2.0,0.108932,0.820261,52.0,65.0,0.800000
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100,5.0,0.163052,0.788332,14.0,65.0,0.215385


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [13]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.549     0.345     0.424      9389
           1      0.687     0.835     0.754     16162

    accuracy                          0.655     25551
   macro avg      0.618     0.590     0.589     25551
weighted avg      0.636     0.655     0.633     25551



Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


In [21]:
# Experiment 1: 90-day feature window

features_90d = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
    ),

    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            -- Most recent 30 days
            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_last30,

            -- Previous 30 days
            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 60 DAY
                     AND f.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev30,

            -- Older 30 days
            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 90 DAY
                     AND f.report_date <= b.end_d - INTERVAL 60 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_old30,

            -- Clicks in most recent 30 days
            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_clicks
                    ELSE 0
                END
            ) AS clk_last30,

            -- Average position in most recent 30 days
            AVG(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_avg_position
                END
            ) AS pos_last30

        FROM {TABLES['fact_daily']} f, bounds b

        WHERE f.report_date > b.end_d - INTERVAL 90 DAY

        GROUP BY 1, 2

        HAVING imp_prev30 >= 100
    )

    SELECT *
    FROM windowed

""").df()

print(f"90-day feature table: {len(features_90d):,} content items")

features_90d.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

90-day feature table: 111,247 content items


,client_hash_id,content_hash_id,imp_last30,imp_prev30,imp_old30,clk_last30,pos_last30
0,client_e547b89c05043229,content_d0fa1bbfbc10caf8,1628.0,2298.0,2070.0,0.0,11.779379
1,client_e547b89c05043229,content_4c1e972bec56132e,7098.0,13712.0,5872.0,44.0,6.959208
2,client_e547b89c05043229,content_64cad58fc02e7605,290.0,251.0,598.0,0.0,48.972195
3,client_e547b89c05043229,content_4e48bd81bb37eb4f,23065.0,21066.0,21394.0,12.0,28.140689
4,client_e547b89c05043229,content_f338440914b1ab00,1460.0,1919.0,2602.0,5.0,20.731918


In [22]:
# Step 2: Add the new historical feature

data_90d = features_90d.merge(
    qsignals,
    on="content_hash_id",
    how="left"
)

# Create the same declining label
data_90d["is_declining"] = (
    data_90d["imp_last30"] < 0.8 * data_90d["imp_prev30"]
).astype(int)

# Features used for the model
feature_cols_90d = [
    "imp_prev30",
    "imp_old30",          # NEW FEATURE
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
]

model_data_90d = data_90d.dropna(subset=feature_cols_90d)

print(f"Rows available for modeling: {len(model_data_90d):,}")

print("\nFeatures used:")
print(feature_cols_90d)

Rows available for modeling: 102,203

Features used:
['imp_prev30', 'imp_old30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']


In [23]:
# Step 3: Random train/test split

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X = model_data_90d[feature_cols_90d]
y = model_data_90d["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

random_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

random_model.fit(X_train, y_train)

random_pred = random_model.predict(X_test)

print("RANDOM SPLIT")
print(classification_report(y_test, random_pred))

RANDOM SPLIT
              precision    recall  f1-score   support

           0       0.66      0.41      0.51      9389
           1       0.72      0.88      0.79     16162

    accuracy                           0.71     25551
   macro avg       0.69      0.64      0.65     25551
weighted avg       0.70      0.71      0.69     25551



In [24]:
# Step 4: GroupShuffleSplit by client

from sklearn.model_selection import GroupShuffleSplit

X = model_data_90d[feature_cols_90d]
y = model_data_90d["is_declining"]
groups = model_data_90d["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

group_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

group_model.fit(X_train_group, y_train_group)

group_pred = group_model.predict(X_test_group)

print("GROUP SHUFFLE SPLIT — CLIENT LEVEL")
print(classification_report(y_test_group, group_pred))

GROUP SHUFFLE SPLIT — CLIENT LEVEL
              precision    recall  f1-score   support

           0       0.49      0.50      0.49     16706
           1       0.76      0.75      0.75     34996

    accuracy                           0.67     51702
   macro avg       0.62      0.62      0.62     51702
weighted avg       0.67      0.67      0.67     51702



In [26]:
# Step 5: Feature importance

import pandas as pd

importance_df = pd.DataFrame({
    "feature": feature_cols_90d,
    "importance": group_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("FEATURE IMPORTANCE — CLIENT GROUP MODEL")
print(importance_df.to_string(index=False))

FEATURE IMPORTANCE — CLIENT GROUP MODEL
        feature  importance
      imp_old30    0.202421
     anon_share    0.185191
     rare_share    0.184724
     imp_prev30    0.168609
top_query_share    0.148319
visible_queries    0.110736


### Conclusion / Interpretation

The 90-day experiment improved the feature set by adding historical impression trends through `imp_old30`. The random split achieved **71% accuracy and 0.69 weighted F1**, while the client-level `GroupShuffleSplit` achieved **67% accuracy and 0.67 weighted F1**.

The lower performance on unseen clients indicates that the model performs better when the same clients can appear in both training and testing data. Therefore, the client-level split provides a more realistic estimate of generalization to new clients.

Among the features, `imp_old30` had the highest feature importance (0.202), followed by `anon_share` (0.185) and `rare_share` (0.185). This suggests that historical impression behavior and query-mix characteristics were important signals for predicting content decline in this experiment.

Overall, the experiment shows why **group-based evaluation is important** when multiple content items belong to the same client.
